In [ ]:
import pandas as pd
from datetime import datetime


In [68]:

df_hydro = pd.read_excel("/Users/6256481/Code/river-hierarchy/SWORD_gauge_match/outputs/mask_hydro_quality.xlsx", sheet_name = 'Hydrograph_quality')
df_hydro = df_hydro[(~df_hydro['Peak_quality'].isin(['Bad', '-'])) & (~df_hydro['Peak_match'].isin(['Bad', '-']))]
df_hydro = df_hydro[['Example', 'Up', 'Period_start', 'Period_end', 'Orientation', 'Forcing_time','mask_file']]

In [59]:
df_hydro[df_hydro['Example'] == 2]

,Example,Up,Period_start,Period_end,Orientation,Forcing_time,mask_file
0,2,CL:3179520,2025-07-26 00:01:00+0000,2026-08-10 00:00:00+0000,SN,18.0,example_2_2025_summer_NH_any_water_uint8.tif


In [78]:
ex_id = 56
row = df_hydro[df_hydro['Example'] == ex_id].iloc[0]

if ex_id < 10:
    ex_id = f'0{ex_id}'
mask_id = ex_id



orientation = row['Orientation']
# 2 EW
# 3 SN

# Level detection Compound
nv = 'compound-bubbles' 
nv = 'independent-units'
if nv == 'compound-bubbles':
    nv_ = 'compound_bubbles'
    nvs = 'compound_bubbles'

elif nv == 'independent-units':
    nv_ = 'independent_units'
    nvs = 'indep'

anchor = True
if anchor == True:
    anchored = 'anchored'
else:
    anchored = 'realized'

def change_timestamp(dt_str):
  dt = datetime.strptime(dt_str, "%Y-%m-%d %H:%M:%S%z")
  return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

STATION_KEY = row['Up']
start_time  = change_timestamp(row['Period_start'])
end_time    = change_timestamp(row['Period_end'])
forcing_time = int(row['Forcing_time'])

################################################
# Mask prep: Create binary mask from selected mask
################################################
mask_prep_string = rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python -B \
  rivgraph_centerline/run_workflow.py prepare-mask \
  --name {ex_id} \
  --source-mask /Volumes/PhD/river_hierarchy/data/water_masks/{row['mask_file']}"""
print("Mask prep: create binary mask with correct crs\n")
print(mask_prep_string)
print('\n\n')
################################################
# Apply edits from created polygons:
# - polygon file: "example_name"_manual_edits
# - add "action" field: commands = add, remove
# - crs: crs of prepped file
################################################
apply_edits_string = rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python -B \
  rivgraph_centerline/run_workflow.py apply-edits \
  --name {ex_id}"""
print("Apply mask edits\n")
print(apply_edits_string)
print('\n\n')
################################################
# create base centerline file
################################################
run_rivgraph = rf"""MPLCONFIGDIR=/tmp/matplotlib-rivgraph \
  /opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python -B \
  rivgraph_centerline/run_workflow.py run-rivgraph \
  --name {ex_id} \
  --exit-sides {orientation} \
  --run-dir rivgraph_centerline/outputs/runs/{ex_id} \
  --assign-flow"""
print("Run base rivgraph\n")
print(run_rivgraph)
print('\n\n')


################################################
# unit detection
################################################
level_detection_string = rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python \
  network_variants/run_collapse_experiment.py \
  {nv} \
  --cleaned-mask rivgraph_centerline/outputs/runs/{ex_id}/masks_cleaned/{ex_id}_cleaned.tif \
  --reviewed-links /Users/6256481/Code/river-hierarchy/rivgraph_centerline/outputs/runs/{ex_id}/rivgraph/{ex_id}_links.gpkg \
  --reviewed-nodes /Users/6256481/Code/river-hierarchy/rivgraph_centerline/outputs/runs/{ex_id}/rivgraph/{ex_id}_nodes.gpkg \
  --exit-sides {orientation} \
  --experiment-id {ex_id}_{nv_} \
  --output-dir network_variants/output/{ex_id}_{nvs} \
  --sword-node-source /Volumes/PhD/SWORD/v17c/ \
  --sword-example-stations-source SWORD_gauge_match/outputs/hierarchy_examples_filtered_subdaily_manual_updates_final.gpkg \
  --sword-station-match-source SWORD_gauge_match/outputs/selected_event_stations_same_main_path.gpkg \
  --sword-reach-buffer-steps 2 \
  --sword-wse-field wse_obs_p50 \
  --compute-equivalent-resistance \
  --equivalent-resistance-law chezy \
  --equivalent-geometry-method wide \
  --equivalent-chezy-cz 22 \
  --equivalent-kb-mode based_reference_section \
  --equivalent-forcing-path SWORD_gauge_match/outputs/subdaily_values/{STATION_KEY[0:2]}/subdaily_timeseries.parquet \
  --equivalent-forcing-station-key {STATION_KEY} \
  --equivalent-station-key-column station_key \
  --equivalent-time-column time \
  --equivalent-discharge-column discharge \
  --equivalent-forcing-start-time {start_time} \
  --equivalent-forcing-end-time {end_time} \
  --equivalent-forcing-resample-minutes 18 \
  --equivalent-forcing-output-cache-dir SWORD_gauge_match/outputs/rapid_forcing_cache \
  --equivalent-kb-model-path RAPID/assets/based_model_v2.ubj \
  --equivalent-kb-width-sample-field width_wet \
  --equivalent-kb-width-percentile 90 \
  --collapse-corridor-mode both
"""
print("Unit detection\n")
print(level_detection_string)
print('\n\n')


################################################
# K calculations
################################################
k_equivalent = rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python \
    network_variants/run_equivalent_resistance_experiment.py \
    /Users/6256481/Code/river-hierarchy/network_variants/output/{ex_id}_{nvs}__{anchored} \
    --equivalent-resistance-law chezy \
    --equivalent-geometry-method wide \
    --equivalent-chezy-cz 22 \
    --equivalent-kb-mode based_reference_section \
    --equivalent-forcing-path SWORD_gauge_match/outputs/subdaily_values/{STATION_KEY[0:2]}/subdaily_timeseries.parquet \
    --equivalent-forcing-station-key {STATION_KEY} \
    --equivalent-station-key-column station_key \
    --equivalent-time-column time \
    --equivalent-discharge-column discharge \
    --equivalent-forcing-start-time {start_time} \
    --equivalent-forcing-end-time {end_time} \
    --equivalent-forcing-resample-minutes {forcing_time} \
    --equivalent-forcing-output-cache-dir SWORD_gauge_match/outputs/rapid_forcing_cache \
    --equivalent-kb-model-path RAPID/assets/based_model_v2.ubj \
    --equivalent-kb-width-sample-field width_wet \
    --equivalent-kb-width-percentile 90"""
print("K equivalent calculation\n")
print(k_equivalent)
print('\n\n')

################################################
# Run Rapid prep
################################################
base_rapid_prep = rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python \
  RAPID/run_prepare_experiment.py \
  /Users/6256481/Code/river-hierarchy/network_variants/output/{ex_id}_{nvs}__{anchored} \
  --forcing-path SWORD_gauge_match/outputs/subdaily_values/{STATION_KEY[0:2]}/subdaily_timeseries.parquet \
  --forcing-station-key {STATION_KEY} \
  --station-key-column station_key \
  --time-column time \
  --discharge-column discharge \
  --forcing-start-time {start_time} \
  --forcing-end-time {end_time} \
  --forcing-resample-minutes {forcing_time} \
  --forcing-output-cache-dir SWORD_gauge_match/outputs/rapid_forcing_cache \
  --kb-mode based_reference_section \
  --kb-model-path RAPID/assets/based_model_v2.ubj \
  --kb-width-sample-field width_wet \
  --kb-width-percentile 90 \
  --use-celerity-capping \
  --min-celerity-mps 0.05 \
  --max-celerity-mps 2.524 \
  --target-subreach-length-m 200 \
  --min-effective-length-for-k-m 200 \
  --max-slope-for-k 0.005 \
  --section-slope-ratio-max 5 \
  --vary-width-by-subreach \
  --subreach-width-sample-field width_wet"""
print("RAPID prep\n")
print(base_rapid_prep)
print('\n\n')
################################################
# Run Rapid
################################################
run_rapid_string = rf"""/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python \
  RAPID/run_rapid_experiment.py \
  /Users/6256481/Code/river-hierarchy/network_variants/output/{ex_id}_{nvs}__{anchored}"""
print('RAPID\n')
print(run_rapid_string)






Mask prep: create binary mask with correct crs

/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python -B \
  rivgraph_centerline/run_workflow.py prepare-mask \
  --name 56 \
  --source-mask /Volumes/PhD/river_hierarchy/data/water_masks/example_56_2025_summer_episodic_mask_uint8.tif



Apply mask edits

/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python -B \
  rivgraph_centerline/run_workflow.py apply-edits \
  --name 56



Run base rivgraph

MPLCONFIGDIR=/tmp/matplotlib-rivgraph \
  /opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python -B \
  rivgraph_centerline/run_workflow.py run-rivgraph \
  --name 56 \
  --exit-sides WN \
  --run-dir rivgraph_centerline/outputs/runs/56 \
  --assign-flow



Unit detection

/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python \
  network_variants/run_collapse_experiment.py \
  independent-units \
  --cleaned-mask rivgraph_centerline/outputs/runs/56/masks_cleaned/56_cleaned.tif \
  --reviewed-links /Users/6256481/Code/river-hierarchy/rivgr

In [47]:
import rasterio
mask_prep = f'/Users/6256481/Code/river-hierarchy/rivgraph_centerline/outputs/runs/{ex_id}/masks_prepared/{ex_id}_binary_projected.tif'


with rasterio.open(mask_prep) as src:
    print("CRS as EPSG:", src.crs.to_epsg() if src.crs else None)

CRS as EPSG: 32617
